In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../data/raw/nyc_sep.csv', encoding='utf-8')
df.shape
df.head(5)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1352055619826961511,https://www.airbnb.com/rooms/1352055619826961511,20260914002229,2026-09-14,NaN,"New 3 bed, 1.5 bath",This stylish place to stay is perfect for grou...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,677651543,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN
1,1215930899551057941,https://www.airbnb.com/rooms/1215930899551057941,20260914002229,2026-09-14,NaN,Queens apartament,Bring the whole family to this great place wit...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,392517036,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
2,1203866180302079266,https://www.airbnb.com/rooms/1203866180302079266,20260914002229,2026-09-15,NaN,Luxury Meets Comfort Stay NYC,This stylish place is close to must-see destin...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,115082911,...,NaN,NaN,NaN,NaN,NaN,2,0,2,0,NaN
3,15310580,https://www.airbnb.com/rooms/15310580,20260914002229,2026-09-15,previous scrape,Nice apartment in Gramercy.,Nice apartment in one of the best neighbors of NY,NaN,https://a0.muscache.com/pictures/c2c78aad-78c9...,27336207,...,4.88,5.00,4.25,NaN,NaN,2,2,0,0,0.07
4,32842295,https://www.airbnb.com/rooms/32842295,20260914002229,2026-09-15,previous scrape,Beautiful bedroom and full bathroom,NaN,NaN,https://a0.muscache.com/pictures/b946d6e3-07d6...,233227636,...,4.89,4.78,4.75,NaN,NaN,1,0,1,0,0.40


In [4]:
df.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_profile_id', 'host_profile_url', 'host_name',
       'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months',
       'hosts_time_as_host_years', 'hosts_time_as_host_months',
       'host_location', 'host_about', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
       'host_listings_count', 'host_total_listings_count',
       'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'price_quote_checkin_date', 'price_quote_c

In [5]:
df.shape

(29918, 90)

In [17]:
df[['review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value']]

,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.38,4.88,3.88,4.75,4.88,5.00,4.25
4,4.50,4.78,4.22,4.97,4.89,4.78,4.75
...,...,...,...,...,...,...,...
29913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29914,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29915,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29916,5.00,5.00,5.00,5.00,5.00,5.00,5.00


## Requirements for data cleaning
`host_identity_verified` should be t,
`price_quote_price_per_night` should not be NA,
`has_availability` should be t,
active referring to `estimated_occupancy_l365d` not 0 or `availability_365` (availability for next 365 days) not 0




In [6]:
verified = df['host_identity_verified'] == 't'
has_price = df['price_quote_price_per_night'].notna()
available = df['has_availability'] == 't'
active = ((df['estimated_occupancy_l365d'] != 0) | (df['availability_365'] != 0))

df_clean = df[verified & has_price & available & active]
df_clean.shape

(18460, 90)

In [7]:
df_clean.loc[df_clean['maximum_nights'] < df_clean['minimum_nights'], ['accommodates', 'beds', 'bedrooms']]

,accommodates,beds,bedrooms


In [8]:
df_clean['review_scores_rating'].describe()

count    13661.000000
mean         4.724584
std          0.469595
min          1.000000
25%          4.670000
50%          4.860000
75%          5.000000
max          5.000000
Name: review_scores_rating, dtype: float64

In [19]:
df_analysis = df_clean[['id', 'name', 'host_id', 'host_name', 'host_is_superhost', 'host_listings_count', 'neighbourhood_cleansed', 
    'neighbourhood_group_cleansed', 'latitude', 'longitude', 'room_type', 'accommodates', 'price_quote_checkin_date', 'price_quote_checkout_date',
    'price_quote_price_per_night', 'minimum_nights', 'number_of_reviews', 'number_of_reviews_l30d', 'review_scores_rating', 
    'reviews_per_month', 'availability_30', 'calendar_last_scraped', 'calculated_host_listings_count', 'estimated_occupancy_l365d', 
    'estimated_revenue_l365d']]

df_analysis


,id,name,host_id,host_name,host_is_superhost,host_listings_count,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,...,minimum_nights,number_of_reviews,number_of_reviews_l30d,review_scores_rating,reviews_per_month,availability_30,calendar_last_scraped,calculated_host_listings_count,estimated_occupancy_l365d,estimated_revenue_l365d
1,1215930899551057941,Queens apartament,392517036,Roman,f,1,Maspeth,Queens,40.721708,-73.907840,...,31.0,0,0,NaN,NaN,29,2026-09-14,1,0,0.0
7,791539646807027059,The studio in the heights,493039547,Manuel,f,1,Washington Heights,Manhattan,40.854703,-73.938423,...,30.0,0,0,NaN,NaN,30,2026-09-14,1,0,0.0
8,984905638210475309,Beach vibes in New York City,538113856,Isabel,f,1,Maspeth,Queens,40.726240,-73.905540,...,30.0,0,0,NaN,NaN,30,2026-09-14,1,0,0.0
9,42220146,nice privet room,291026998,Ghenadie,f,4,Sheepshead Bay,Brooklyn,40.600260,-73.957930,...,30.0,0,0,NaN,NaN,29,2026-09-14,4,0,0.0
10,42220001,NICE AND COZY STUDIO,291026998,Ghenadie,f,4,Sheepshead Bay,Brooklyn,40.600350,-73.959570,...,30.0,0,0,NaN,NaN,29,2026-09-14,4,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29871,1361005144321464086,RoveTravel | Downing Court | Condo w/ OutdoorS...,17775359,Rove Travel,f,131,West Village,Manhattan,40.729249,-74.004753,...,180.0,0,0,NaN,NaN,15,2026-09-15,103,0,0.0
29872,1364308392617365525,Spacious One Bdrm in Brooklyn,456589439,House Lynk,f,2,Crown Heights,Brooklyn,40.674770,-73.913620,...,90.0,0,0,NaN,NaN,0,2026-09-15,2,0,0.0
29892,1653803229211742593,"Superior Studio Apt, 1 Queen, 1 Sofa Bed - FiDi",753417675,Nester Finn,t,14,Financial District,Manhattan,40.707529,-74.006188,...,1.0,5,0,4.6,1.15,0,2026-09-15,14,64,151360.0
29903,1403073945699702413,Special Longterm Discount Luxury Furnished 1BR,15145088,Izi,t,11,Upper West Side,Manhattan,40.787490,-73.973130,...,60.0,0,0,NaN,NaN,27,2026-09-15,11,0,0.0


In [10]:
df_analysis['price_quote_checkin_date'] = pd.to_datetime(df_analysis['price_quote_checkin_date'], format='%Y-%m-%d')
df_analysis['price_quote_checkin_date'].head(5)

1    2026-09-15
7    2026-09-14
8    2026-09-14
9    2026-09-15
10   2026-09-15
Name: price_quote_checkin_date, dtype: datetime64[us]

In [11]:
# Creating a week aggregation
df_analysis['checkin_week'] = df_analysis['price_quote_checkin_date'].dt.to_period('W').dt.start_time
df_analysis['checkin_week'].value_counts().sort_values(ascending=False).head(20)

checkin_week
2026-09-14    6431
2026-09-28    1911
2026-09-21    1426
2026-10-26    1145
2026-12-28     887
2026-10-05     739
2026-10-12     732
2026-11-02     660
2026-11-30     588
2026-10-19     465
2026-12-14     420
2026-11-09     342
2026-11-16     275
2026-12-21     263
2026-11-23     231
2026-12-07     193
2027-01-04     193
2027-02-01     151
2027-03-29     139
2027-05-31     122
Name: count, dtype: int64

In [12]:
# creating a function for week aggregation:
def get_week(df, date_col = 'price_quote_checkin_date'):
    df['checkin_week'] = pd.to_datetime(df[date_col]).dt.to_period('W').dt.start_time

In [13]:
df_analysis.groupby('checkin_week').agg(
    ave_price=('price_quote_price_per_night', 'mean')
).sort_values(by='checkin_week', ascending=True).head(20)

,ave_price
checkin_week,
2026-09-14,407.002723
2026-09-21,458.997973
2026-09-28,222.671544
2026-10-05,203.118390
2026-10-12,230.857664
2026-10-19,230.916796
2026-10-26,195.303336
2026-11-02,238.169470
2026-11-09,213.405146


In [14]:
df_analysis['price_quote_per_night_per_person'] = df_analysis['price_quote_price_per_night'] / df_analysis['accommodates']
df_analysis['price_quote_per_night_per_person'].describe()

count    18460.000000
mean       108.219141
std        184.581222
min          4.178750
25%         46.624583
50%         70.956667
75%        116.273750
max      11028.870000
Name: price_quote_per_night_per_person, dtype: float64

In [15]:
df_analysis.groupby(['neighbourhood_group_cleansed'])['review_scores_rating'].describe()

,count,mean,std,min,25%,50%,75%,max
neighbourhood_group_cleansed,,,,,,,,
Bronx,572.0,4.741066,0.405848,1.0,4.6700,4.845,4.9925,5.0
Brooklyn,4825.0,4.764170,0.439971,1.0,4.7200,4.880,5.0000,5.0
Manhattan,5530.0,4.677143,0.510364,1.0,4.5900,4.820,5.0000,5.0
Queens,2504.0,4.742081,0.447646,1.0,4.6700,4.860,5.0000,5.0
Staten Island,230.0,4.803304,0.298258,2.0,4.7325,4.880,5.0000,5.0
